# ?? SHE Full-Song Film Master Render (7:25 | 10,680 Frames | 1080p)
### Resumable 10-Chunk Pipeline, Pre-Render Quality Gates, & Google Drive Checkpointing

**Canonical project:** [`sayan-bhattacharya/aianimationautomate`](https://github.com/sayan-bhattacharya/aianimationautomate). SHE is maintained here as a standalone music-film project. It is separate from the NOIR film repository and does not depend on NOIR code or assets.

This render incorporates the reviewed scene revisions: mushroom scale and grounded roots, a slower tree sway, a dinosaur aerial on the opening CRT feed, restrained analog noise on the far-right CRT feed, quieter water reflections, the 7:05 Buddha/CRT placement, and the 7:21 `.STORE` end-card timing.

Run notebook cells in order. The full-length 1080p master has not been rendered by this notebook yet.


## Step 1: Install Dependencies & Setup Environment
Installs `ffmpeg`, `opencv-python`, `librosa`, `Pillow`, and verifies hardware acceleration.

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q opencv-python pillow numpy librosa

import os
import sys
import multiprocessing as mp
import subprocess

print(f"✓ CPU cores available: {mp.cpu_count()}")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "Running on high-core CPU/TPU instance"


## Step 1b: Mount Google Drive (Recommended)
Mount your Google Drive to store completed chunks and the final 1080p master video directly in your Drive. If Colab disconnects, your progress is permanently preserved.

In [ ]:
# --- MOUNT GOOGLE DRIVE & SETUP DIRECTORY ---
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create dedicated project render folder in your Google Drive root
DRIVE_RENDER_DIR = '/content/drive/MyDrive/SHE_Film_Render_2026-11-25_rev1'
DRIVE_CHUNKS_DIR = os.path.join(DRIVE_RENDER_DIR, 'chunks')
os.makedirs(DRIVE_CHUNKS_DIR, exist_ok=True)

print(f"✓ Google Drive mounted!")
print(f"✓ Render output & backup directory: {DRIVE_RENDER_DIR}")


## Step 2: Pull SHE Code and Assets from the Canonical Repository
The notebook, renderer, prepared sprites, master audio, and audio features all come from `aianimationautomate` `main`. This keeps the Colab render self-contained and independent of the separate NOIR project.


In [ ]:
import os
import sys
import shutil
import subprocess
import importlib

REPO_DIR = '/content/aianimationautomate'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
print('Cloning canonical SHE project: aianimationautomate/main...')
subprocess.run([
    'git','clone','--depth','1','--branch','main','--single-branch',
    'https://github.com/sayan-bhattacharya/aianimationautomate.git',
    REPO_DIR
],check=True)

required = [
    os.path.join(REPO_DIR, 'she_film'),
    os.path.join(REPO_DIR, 'assets'),
    os.path.join(REPO_DIR, 'She.wav'),
    os.path.join(REPO_DIR, 'audio_features_24fps.npy'),
]
missing = [path for path in required if not os.path.exists(path)]
if missing:
    raise FileNotFoundError('Canonical repository is missing required render inputs: ' + ', '.join(missing))

for source, target in [
    (os.path.join(REPO_DIR, 'she_film'), '/content/she_film'),
    (os.path.join(REPO_DIR, 'assets'), '/content/assets'),
]:
    if os.path.exists(target):
        shutil.rmtree(target)
    shutil.copytree(source, target)
shutil.copy2(os.path.join(REPO_DIR, 'She.wav'), '/content/She.wav')
shutil.copy2(os.path.join(REPO_DIR, 'audio_features_24fps.npy'), '/content/audio_features_24fps.npy')

while '/content' in sys.path:
    sys.path.remove('/content')
sys.path.insert(0, '/content')
for name in list(sys.modules):
    if name == 'she_film' or name.startswith('she_film.'):
        del sys.modules[name]
importlib.invalidate_caches()

from she_film import clock, generate_screens_cache, plates
if abs(clock.DURATION - 445.0) > 1e-6 or clock.TOTAL_FRAMES != 10680:
    raise RuntimeError(f'Loaded repository code does not match this 7:25 notebook: DURATION={clock.DURATION}, TOTAL_FRAMES={clock.TOTAL_FRAMES}')

# Build the directed dinosaur aerial and restrained far-right analog-noise feed.
active_package = os.path.dirname(os.path.abspath(generate_screens_cache.__file__))
plates_package = os.path.dirname(os.path.abspath(plates.__file__))
if os.path.realpath(active_package) != os.path.realpath(plates_package):
    raise RuntimeError(f'Package path mismatch: generator={active_package}, plates={plates_package}')
screens_path = generate_screens_cache.build_screens_cache(plates.CACHE)
import numpy as np
with np.load(screens_path) as screens:
    for key in ('scr_left','scr_mid_left','scr_mid_right','scr_right'):
        if key not in screens or screens[key].ndim != 4 or len(screens[key]) < 240 or screens[key].shape[1:] != (320,240,3):
            raise RuntimeError(f'Invalid or missing screen feed: {key}')
        movement = float(np.mean(np.abs(screens[key][0].astype(np.float32)-screens[key][len(screens[key])//2].astype(np.float32))))
        if movement <= 2.0:
            raise RuntimeError(f'Static CRT feed: {key}; midpoint difference={movement:.2f}')
print('Canonical repository:', REPO_DIR)
print('SHE renderer:', active_package)
print('Prepared assets:', os.path.join(REPO_DIR, 'assets'))
print('Master audio: /content/She.wav')
print('Audio features: /content/audio_features_24fps.npy')
print('Four moving CRT feeds validated:', screens_path)


## Art Direction and Scene Coupling

Keep the camera and horizon quiet. Let only a few low-contrast water crests move, and treat light as a gradual shift in perception. The dinosaurs should read as distant, monumental presences. Plant forms emerge slowly with the music, hold long enough to feel calm, then recede; avoid cutting on every beat.

The ending preserves the requested story order: at approximately **7:05**, the Buddha tableau shows the dinosaur aerial on the first CRT screen and subdued analog noise at the far right; at approximately **7:21**, the closing credit includes `.STORE`.

This is a visual direction note for SHE only. The NOIR film has its own repository and materials.


## Step 3: Run Pre-Render Quality Gatekeeper Review
**MANDATORY**: Before spending compute on the full render, this automated reviewer audits all 7 film segments, testing lifecycle timing, pterodactyl continuity, celestial lighting/reflections, CRT placement, and four moving CRT feeds. Code checks catch timing and asset defects; they do not replace visual review. Rendering is blocked if any hard check fails.


In [ ]:
from she_film import pre_render_code_review

passed = pre_render_code_review.run_pre_render_code_review()
from IPython.display import display, Image as NotebookImage
debug_image = '/content/preview_720p/tv_screen_alignment_debug.png'
if os.path.exists(debug_image):
    display(NotebookImage(filename=debug_image))
if not passed:
    raise RuntimeError('❌ PRE-RENDER GATEKEEPER FAILED: Code review found defects. Halting execution.')
print('\n>>> ALL PRE-RENDER CHECKS PASSED: Cleared for 1080p Master Render! <<<')


## Step 4: Full-Song 1080p Parallel Render Pipeline
Renders the complete 10,680 frames in 10 sequential, verified chunks to keep memory use predictable. The chunk selector rebuilds only changed intervals and reuses verified unaffected chunks; each completed render is backed up to Google Drive.

In [ ]:
import cv2
import time
from she_film import frame, plates
from she_film.clock import DURATION, FPS, TOTAL_FRAMES

OUTPUT_DIR = '/content/she_render_1080p_2026-11-25_rev1'
CHUNKS_DIR = os.path.join(OUTPUT_DIR, 'chunks')
os.makedirs(CHUNKS_DIR, exist_ok=True)

# Google Drive backup path if mounted
DRIVE_BACKUP = os.path.join(DRIVE_RENDER_DIR, 'chunks') if os.path.exists('/content/drive/MyDrive') else None
if DRIVE_BACKUP:
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    print(f"✓ Google Drive live backup active: {DRIVE_BACKUP}")

NUM_CHUNKS = 10
# All ten chunks are affected by this pass. For later edits, use only overlapping chunk IDs.
CHUNKS_TO_RENDER = set(range(1, NUM_CHUNKS + 1))

# Calculate exact chunk frame boundaries
chunk_ranges = []
chunk_sz = TOTAL_FRAMES // NUM_CHUNKS
for i in range(NUM_CHUNKS):
    f_start = i * chunk_sz
    f_end = (i + 1) * chunk_sz if i < NUM_CHUNKS - 1 else TOTAL_FRAMES
    chunk_ranges.append((i + 1, f_start, f_end))

print(f"Planned {NUM_CHUNKS} verified chunks totaling {TOTAL_FRAMES} frames @ 24fps (1920x1080):")
for c_id, f0, f1 in chunk_ranges:
    print(f"  Chunk {c_id:02d}: Frames {f0:05d} -> {f1:05d} ({f1 - f0} frames, {f0/24.0:.1f}s -> {f1/24.0:.1f}s)")

# Load plates into memory
print("\nLoading shared plates...")
p = plates.load()
print("✓ Plates loaded successfully.")

def render_chunk(c_info):
    c_id, f_start, f_end = c_info
    n_chunk_frames = f_end - f_start
    chunk_path = os.path.join(CHUNKS_DIR, f"chunk_{c_id:02d}.mp4")
    drive_chunk = os.path.join(DRIVE_BACKUP, f"chunk_{c_id:02d}.mp4") if DRIVE_BACKUP else None
    
    # Unaffected chunks can be restored from Drive or reused locally.
    if c_id not in CHUNKS_TO_RENDER:
        candidates = [drive_chunk, chunk_path]
        reusable = None
        for candidate in candidates:
            if candidate and os.path.exists(candidate):
                cap = cv2.VideoCapture(candidate)
                existing = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.release()
                if existing == n_chunk_frames:
                    reusable = candidate
                    break
        if reusable is None:
            raise FileNotFoundError(f'Chunk {c_id:02d} is marked unchanged but no verified cached copy exists.')
        if reusable != chunk_path:
            shutil.copy2(reusable, chunk_path)
        print(f'REUSED unchanged Chunk {c_id:02d}: {chunk_path}')
        return chunk_path

    # Drive cache check: restore from Google Drive if already rendered previously
    # Versioned output and Drive folders isolate this code revision. Resume only
    # complete chunks produced in this folder; incomplete chunks are regenerated.
    FORCE_FRESH_RENDER = False
    if not FORCE_FRESH_RENDER and drive_chunk and os.path.exists(drive_chunk):
        shutil.copy2(drive_chunk, chunk_path)
        print(f"✓ Chunk {c_id:02d} restored from Google Drive!")
        return chunk_path
        
    if not FORCE_FRESH_RENDER and os.path.exists(chunk_path):
        cap = cv2.VideoCapture(chunk_path)
        existing = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        if existing >= n_chunk_frames - 1:
            print(f"✓ Chunk {c_id:02d} already complete ({existing} frames) in local cache. Skipping.")
            return chunk_path
            
    print(f"\n▶ Rendering Chunk {c_id:02d} ({f_start} -> {f_end}, {n_chunk_frames} frames)...")
    video_cmd = ['ffmpeg','-y','-loglevel','error','-f','rawvideo','-vcodec','rawvideo','-s','1920x1080','-pix_fmt','rgb24','-r',str(FPS),'-i','-','-an','-c:v','libx264','-preset','ultrafast','-crf','18','-pix_fmt','yuv420p','-movflags','+faststart',chunk_path]
    log_handle = open(chunk_path + '.ffmpeg.log','w')
    writer = subprocess.Popen(video_cmd,stdin=subprocess.PIPE,stderr=log_handle)
    t0 = time.time()
    for idx in range(n_chunk_frames):
        cur_frame_idx = f_start + idx
        t_cur = cur_frame_idx / FPS
        f_arr = frame.render(p, t_cur)
        writer.stdin.write(f_arr.tobytes())
        if (idx + 1) % 200 == 0 or idx == n_chunk_frames - 1:
            pct = (idx + 1) / n_chunk_frames * 100
            fps_rate = (idx + 1) / max(0.01, time.time() - t0)
            print(f"  Chunk {c_id:02d}: {idx+1}/{n_chunk_frames} ({pct:.1f}%) | {fps_rate:.1f} fps")
    writer.stdin.close()
    encode_status = writer.wait()
    log_handle.close()
    if encode_status != 0:
        raise RuntimeError(f'ffmpeg failed for chunk {c_id:02d}; inspect ' + chunk_path + '.ffmpeg.log')
    
    # Auto-copy completed chunk to Google Drive
    if drive_chunk:
        shutil.copy2(chunk_path, drive_chunk)
        print(f"  ↳ Backed up Chunk {c_id:02d} to Google Drive.")
        
    print(f"✓ Chunk {c_id:02d} complete!")
    return chunk_path

rendered_chunks = []
for c_info in chunk_ranges:
    path = render_chunk(c_info)
    rendered_chunks.append(path)

print("\n=======================================================")
print(f"✓ ALL {NUM_CHUNKS} CHUNKS RENDERED AND BACKED UP SUCCESSFULLY!")
print("=======================================================")


## Step 5: Master Audio Muxing, Concatenation & SHE Crystal Nocturne Finish
Stitches the 10 verified H.264 chunks with video stream copy, muxes `She.wav` with the exact 7:25 fade, and writes the 1080p master directly into your Google Drive without a second full-length video encode.

The final cell also builds a 50-second **16:9 landscape** Instagram teaser from the rendered master. It preserves the wide composition by request; vertical Reels playback will show a wide video inside the tall phone screen rather than filling it.


In [ ]:
import os
import glob
import shutil
import subprocess

OUTPUT_DIR = '/content/she_render_1080p_2026-11-25_rev1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Resilient chunk discovery (works even if runtime restarted)
if 'rendered_chunks' not in globals() or not rendered_chunks:
    drive_chunks_dir = '/content/drive/MyDrive/SHE_Film_Render_2026-11-25_rev1/chunks'
    local_chunks_dir = os.path.join(OUTPUT_DIR, 'chunks')
    
    if os.path.exists(drive_chunks_dir) and len(glob.glob(os.path.join(drive_chunks_dir, 'chunk_*.mp4'))) > 0:
        search_dir = drive_chunks_dir
    else:
        search_dir = local_chunks_dir
        
    rendered_chunks = sorted(glob.glob(os.path.join(search_dir, 'chunk_*.mp4')))
    print(f"Discovered {len(rendered_chunks)} chunks in {search_dir}:")
    for ch in rendered_chunks:
        print(f"  - {os.path.basename(ch)}")

if not rendered_chunks:
    raise FileNotFoundError("No rendered chunk MP4 files found in local or Drive directories. Run Step 4 first!")

# 2. Write concat list
concat_list_file = os.path.join(OUTPUT_DIR, "concat_chunks.txt")
with open(concat_list_file, "w") as f:
    for chunk_file in rendered_chunks:
        f.write(f"file '{chunk_file}'\n")

FINAL_MASTER_MP4 = os.path.join(OUTPUT_DIR, "she_full_song_1080p_crystal_nocturne_master.mp4")

# 3. Locate master audio She.wav
AUDIO_WAV = "/content/She.wav"
if not os.path.exists(AUDIO_WAV):
    for alt_wav in ["/content/aianimationautomate/She.wav", os.path.join(DRIVE_RENDER_DIR, 'She.wav')]:
        if os.path.exists(alt_wav):
            AUDIO_WAV = alt_wav
            break

print(f"Using master audio: {AUDIO_WAV}")

# 4. ffmpeg lossless stream concat & high-fidelity AAC mux
cmd_concat = [
    "ffmpeg", "-y",
    "-f", "concat", "-safe", "0", "-i", concat_list_file,
    "-i", AUDIO_WAV,
    "-c:v", "copy",
    "-c:a", "aac", "-b:a", "320k",
    "-t", f"{DURATION:.3f}",
    "-af", f"afade=t=in:ss=0:d=1.2,afade=t=out:st={DURATION-12:.3f}:d=12",
    "-shortest",
    FINAL_MASTER_MP4
]

print("Stitching chunks and muxing master audio with ffmpeg...")
subprocess.run(cmd_concat, check=True)

print(f"\n✓ Master video assembled at: {FINAL_MASTER_MP4}")
print(f"  File size: {os.path.getsize(FINAL_MASTER_MP4) / 1024 / 1024:.2f} MB")

# 5. Backup final master directly to Google Drive
if os.path.exists('/content/drive/MyDrive'):
    DRIVE_MASTER = os.path.join(DRIVE_RENDER_DIR, os.path.basename(FINAL_MASTER_MP4))
    print(f"\nCopying final master video to Google Drive: {DRIVE_MASTER}...")
    shutil.copy2(FINAL_MASTER_MP4, DRIVE_MASTER)
    print(f"✓ Successfully saved to Google Drive: {DRIVE_MASTER}!")

# 6. Landscape Instagram teaser (user-requested wide format, not portrait)
# Seven authored excerpts crossfade into a 50-second, 16:9 teaser at 30 fps.
TEASER_MP4 = os.path.join(OUTPUT_DIR, 'SHE_Instagram_Landscape_Teaser_1080p_30fps.mp4')
TEASER_PARTS = [(18.0,7.0),(112.0,8.0),(176.0,8.0),(248.0,8.0),(315.0,8.0),(408.0,8.0),(439.0,6.0)]
TEASER_FADE = 0.5
n_parts = len(TEASER_PARTS)
filter_parts = []
v_sources = ''.join(f'[vsrc{i}]' for i in range(n_parts))
a_sources = ''.join(f'[asrc{i}]' for i in range(n_parts))
filter_parts.append(f'[0:v]split={n_parts}{v_sources}')
filter_parts.append(f'[0:a]asplit={n_parts}{a_sources}')
for i,(start,duration) in enumerate(TEASER_PARTS):
    filter_parts.append(f'[vsrc{i}]trim=start={start}:duration={duration},setpts=PTS-STARTPTS,fps=30,format=yuv420p[v{i}]')
    filter_parts.append(f'[asrc{i}]atrim=start={start}:duration={duration},asetpts=PTS-STARTPTS[a{i}]')
v_prev, a_prev = 'v0', 'a0'
offset = TEASER_PARTS[0][1] - TEASER_FADE
for i in range(1,n_parts):
    v_next = f'vx{i}'
    a_next = f'ax{i}'
    filter_parts.append(f'[{v_prev}][v{i}]xfade=transition=fade:duration={TEASER_FADE}:offset={offset:.3f}[{v_next}]')
    filter_parts.append(f'[{a_prev}][a{i}]acrossfade=d={TEASER_FADE}:c1=tri:c2=tri[{a_next}]')
    v_prev, a_prev = v_next, a_next
    offset += TEASER_PARTS[i][1] - TEASER_FADE
cmd_teaser = [
    'ffmpeg','-y','-loglevel','error','-i',FINAL_MASTER_MP4,
    '-filter_complex',';'.join(filter_parts),
    '-map',f'[{v_prev}]','-map',f'[{a_prev}]',
    '-c:v','libx264','-preset','ultrafast','-crf','18','-pix_fmt','yuv420p',
    '-c:a','aac','-b:a','192k','-movflags','+faststart',TEASER_MP4
]
print('Rendering the landscape 50-second Instagram teaser...')
subprocess.run(cmd_teaser,check=True)
print('Landscape teaser:',TEASER_MP4,'| 1920x1080 | 30 fps | 16:9')
if os.path.exists('/content/drive/MyDrive'):
    DRIVE_TEASER = os.path.join(DRIVE_RENDER_DIR, os.path.basename(TEASER_MP4))
    shutil.copy2(TEASER_MP4, DRIVE_TEASER)
    print('Landscape teaser saved to Google Drive:',DRIVE_TEASER)


## Step 6: Verify Final Master Integrity & Direct Download
Checks frame count (must be exactly 10,680), stream duration, and offers direct browser download.

In [ ]:
import os
import cv2

cap = cv2.VideoCapture(FINAL_MASTER_MP4)
actual_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
actual_fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

print(f"=== FINAL MASTER INTEGRITY AUDIT ===")
print(f"  Expected frames:  {TOTAL_FRAMES}")
print(f"  Rendered frames:  {actual_frames}")
print(f"  FPS:              {actual_fps}")
print(f"  Duration:         {actual_frames / actual_fps:.2f}s (7m 25s)")

actual_duration = actual_frames / actual_fps if actual_fps > 0 else 0.0
if actual_frames == TOTAL_FRAMES and abs(actual_duration - DURATION) <= 0.05:
    print("\n>>> STATUS: EXACT 7:25 MASTER CERTIFIED <<<")
    if os.path.exists(os.path.join(DRIVE_RENDER_DIR, os.path.basename(FINAL_MASTER_MP4))):
        print("✓ Your full 1080p master video is permanently safe in Google Drive ({DRIVE_RENDER_DIR})")
else:
    print(f"\n⚠ Warning: Frame count mismatch ({actual_frames} vs {TOTAL_FRAMES})")

# Optional direct browser download
from google.colab import files
print("Starting browser download of final video...")
files.download(FINAL_MASTER_MP4)


## Step 7: Post-Render Technical Check & Human Visual Review

This cell verifies the encoded stream and extracts a review contact sheet at the film's important transitions. It does **not** certify artistic quality automatically. Inspect the displayed frames for wave prominence, land contact, depth, gradual transformation, silhouette continuity, reflection placement, and the closing mood before treating the master as final.


In [ ]:
import os
import json
import subprocess
import shutil
import cv2
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, Image as NotebookImage

print('=' * 76)
print('POST-RENDER TECHNICAL CHECK + VISUAL REVIEW CONTACT SHEET')
print('=' * 76)
if not os.path.exists(FINAL_MASTER_MP4):
    raise FileNotFoundError(f'Final master not found: {FINAL_MASTER_MP4}')

cap = cv2.VideoCapture(FINAL_MASTER_MP4)
actual_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = float(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = actual_frames / fps if fps > 0 else 0.0
cap.release()
probe = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'stream=codec_type', '-of', 'json', FINAL_MASTER_MP4], check=True, capture_output=True, text=True)
stream_types = [s.get('codec_type') for s in json.loads(probe.stdout).get('streams', [])]
checks = {'frame count': actual_frames == TOTAL_FRAMES, 'duration': abs(duration - DURATION) <= 0.05, 'picture size': (width, height) == (1920, 1080), 'audio stream': 'audio' in stream_types}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'} technical check: {label}")
print(f'  {actual_frames} frames | {fps:.3f} fps | {width}x{height} | {duration:.3f}s | streams={stream_types}')

review_points = [(24.0,'opening mushrooms'),(28.0,'mushroom 2 mature'),(31.9,'mushroom 2 hold ends'),(36.0,'mushroom 2 decay'),(40.1,'small decayed hold'),(54.0,'mushroom grounding'),(120.0,'mushroom hero hold'),(160.0,'arrival glint'),(166.0,'saucer enters'),(178.0,'saucer approach'),(182.5,'saucer landing hold'),(184.9,'landing hold end'),(198.0,'dawn departure'),(204.0,'saucer recedes'),(208.0,'saucer dawn fade'),(210.0,'saucer near vanishing'),(211.0,'saucer fade tail'),(218.0,'forest before dinosaur entrance'),(232.0,'dinosaur entrance'),(260.0,'dinosaur forest hold'),(290.0,'pterodactyl passage'),(391.0,'caravan exit'),(397.0,'forest decay'),(400.0,'bare interval'),(406.0,'mushroom regrowth'),(414.0,'mushroom return'),(424.0,'Buddha transition'),(425.0,'7:05 dinosaur CRT + far-right noise'),(432.0,'CRT screens'),(440.0,'end credits'),(441.0,'7:21 .STORE credits')]
thumbs=[]
cap=cv2.VideoCapture(FINAL_MASTER_MP4)
for t,label in review_points:
    cap.set(cv2.CAP_PROP_POS_MSEC,t*1000); ok,bgr=cap.read()
    if not ok:
        print(f'Could not decode review frame at {t:.1f}s ({label})'); continue
    rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    thumbs.append((t,label,Image.fromarray(rgb).resize((480,270),Image.Resampling.LANCZOS)))
cap.release()
cols=3; cell_w=480; cell_h=304; rows=(len(thumbs)+cols-1)//cols
sheet=Image.new('RGB',(cols*cell_w,rows*cell_h),(16,18,23)); draw=ImageDraw.Draw(sheet)
for i,(t,label,thumb) in enumerate(thumbs):
    x=(i%cols)*cell_w; y=(i//cols)*cell_h
    draw.text((x+8,y+5),f'{t:06.1f}s  {label}',fill=(236,224,195)); sheet.paste(thumb,(x,y+28))
review_sheet_path=os.path.join(OUTPUT_DIR,'final_deliverable_visual_review.jpg')
sheet.save(review_sheet_path,quality=92)
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(DRIVE_RENDER_DIR,exist_ok=True); shutil.copy2(review_sheet_path,os.path.join(DRIVE_RENDER_DIR,os.path.basename(review_sheet_path)))
print(f'Visual review sheet: {review_sheet_path} ({len(thumbs)} frames)')
display(NotebookImage(filename=review_sheet_path))
print('MANUAL ART REVIEW REQUIRED: this contact sheet does not certify composition, depth, transformation quality, wave stillness, or emotional impact.')
if not all(checks.values()): raise RuntimeError('Technical output check failed; inspect the master before distribution.')
